# 04 — Reach: Bi-manual Target Reaching (auxiliary task)

| | |
|---|---|
| **Author** | Narayane Ribeiro Medeiros |
| **Affiliation** | Instituto Tecnológico de Aeronáutica (ITA) |
| **Advisor** | Prof. Marcos Máximo |
| **Context** | Undergraduate's defense — 2026 |

Auxiliary policy trained *before* `02_pick.ipynb`. The G1 humanoid stands and brings its two hands to sampled 3D targets — no walking, no object, no grasp. Once trained, the actor is a cheap warm-start for pick's arm-control burden, avoiding the from-scratch cost of teaching arm coordination while the robot also has to balance.

Task/env/PPO config lives in **`config/policy_reach.py`** (single source of truth for training and inference).

**Domain randomization** on the target pair per reset:
- **Symmetric** targets (`right = mirror(left)`) — ~40 % of resets.
- **Asymmetric** targets (each hand independent) — ~60 % of resets.
- **Heights** uniformly sampled from below-waist (~0.4 m) to above-shoulders (~1.4 m); asymmetric mode allows different heights per hand.

## Notebook flags

Boolean switches for reuse, resume, debug-light mode, and viewer.

In [ ]:
REUSE_TRAINING       = True  # skip training if PPO checkpoint exists
RESUME               = False  # continue from the most recent PPO checkpoint
DEBUG_LIGHT          = True   # True = 64 envs / False = 8192 envs, few iters
RUN_VIEWER           = False  # open viser viewer at localhost:8080

# Environment Setup

Installs pinned dependencies from `config/requirements.txt` via **uv** (5-10x faster
than pip; idempotent skip of satisfied packages), configures MuJoCo for headless
rendering (EGL), and detects the GPU.

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

if sys.version_info[:2] != (3, 10):
    raise RuntimeError(f'Python 3.10 required but got {sys.version.split()[0]}')

os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ['UV_HTTP_TIMEOUT'] = '600'

REQ = Path('../requirements.txt').resolve()
if not REQ.exists():
    raise FileNotFoundError(f'requirements.txt not found at {REQ}')

if shutil.which('uv') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])

subprocess.check_call([
    sys.executable, '-m', 'uv', 'pip', 'install',
    '--index-strategy', 'unsafe-best-match',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu128',
    '-r', str(REQ),
])

import torch, mjlab
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    print(f'GPU count : {n}')
    for i in range(n):
        p = torch.cuda.get_device_properties(i)
        print(f'  [{i}] {p.name}  |  {p.total_memory / 1e9:.1f} GB  |  sm_{p.major}{p.minor}')
    DEVICE = 'cuda:0'
    torch.cuda.set_device(0)
else:
    DEVICE = 'cpu'
    print('WARNING: no GPU detected')

try:
    subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.free',
                    '--format=csv,noheader'], check=False)
except FileNotFoundError:
    pass

# Robot Initialization

Derives `TASK_ID`, G1 joint list, home pose, and reach workspace ranges from
`config/policy_reach.py` so they are printed for quick inspection.

In [ ]:
import sys, os
from pathlib import Path
for _p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_p / 'LICENSE').exists():
        os.chdir(_p); break
sys.path.insert(0, str(Path.cwd()))

from src.policies.policy_reach import (
    TASK_ID,
    G1_JOINTS, G1_HOME_ROOT_POS, G1_HOME_JOINT_POS,
    REACH_X_RANGE, REACH_Y_LEFT_RANGE, REACH_Y_RIGHT_RANGE, REACH_Z_RANGE,
    REACH_SYMMETRY_PROB, REACH_HIT_THRESHOLD,
)

print(f'Task ID          : {TASK_ID}')
print(f'Joints           : {len(G1_JOINTS)}  (derived from mjlab G1 spec)')
print(f'Home root z      : {G1_HOME_ROOT_POS[2]:.3f} m')
print()
print(f'Reach x range    : {REACH_X_RANGE} m  (forward from base)')
print(f'Reach y left     : {REACH_Y_LEFT_RANGE} m')
print(f'Reach y right    : {REACH_Y_RIGHT_RANGE} m')
print(f'Reach z range    : {REACH_Z_RANGE} m  (ground → head)')
print(f'Symmetry prob    : {REACH_SYMMETRY_PROB:.0%}  (fraction of resets w/ mirrored targets)')
print(f'Hit threshold    : {REACH_HIT_THRESHOLD * 100:.1f} cm  (both hands within → +1 bonus)')

## Task Config & Registration

Full MDP (observations, actions, events, rewards, terminations, curriculum) and
the PPO runner config live in `config/policy_reach.py` — single source of truth
for training and inference. Editing rewards, DR ranges, or PPO hyperparameters
means editing that file and re-running this cell.

In [ ]:
from src.policies import policy_reach

# Register the task with mjlab so TrainConfig.from_task / load_env_cfg can find it
policy_reach.register()

_env = policy_reach.g1_reach_env_cfg(play=False)
print(f'Task registered  : {policy_reach.TASK_ID}')
print(f'Entities         : {list(_env.scene.entities.keys())}')
print(f"Sensors          : {len(_env.scene.sensors)}  ({', '.join(s.name for s in _env.scene.sensors)})")
print(f'Events           : {list(_env.events.keys())}')
print(f'Rewards          : {len(_env.rewards)} terms   → {list(_env.rewards.keys())}')
print(f'Terminations     : {list(_env.terminations.keys())}')
print(f'Curriculum       : {list(_env.curriculum.keys())}')
print(f'Sim iter/ls      : {_env.sim.mujoco.iterations}/{_env.sim.mujoco.ls_iterations}')
print(f'Episode length   : {_env.episode_length_s:.2f} s   ({int(_env.episode_length_s / 0.025)} steps @ 40 Hz)')

# Simulation Parameters

Global run settings shared by training and play: task selection, number of parallel environments, PPO hyperparameters.

In [ ]:
from pathlib import Path
import sys; _root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'LICENSE').exists()); import os as _os; _os.chdir(_root); sys.path.insert(0, str(_root))
from src.utils import find_latest_checkpoint

OUT   = Path('outputs'); OUT.mkdir(exist_ok=True)
MEDIA = OUT / 'media';   MEDIA.mkdir(exist_ok=True)

EXPERIMENT_HINT  = 'g1_reach'
LOG_ROOT         = str(OUT / 'logs')

NUM_ENVS         = 8192
MAX_ITERATIONS   = 1000

SIM_TIMESTEP     = 0.005
DECIMATION       = 5
EPISODE_LENGTH_S = 4.0

# PPO / RSL-RL — aux reach uses a shorter horizon than walk
LEARNING_RATE    = 1e-3
CLIP_PARAM       = 0.2
GAMMA            = 0.99
LAMBDA_GAE       = 0.95
ENTROPY_COEF     = 0.01

if DEBUG_LIGHT:
    NUM_ENVS, MAX_ITERATIONS = 64, 3

print(f'Task            : {TASK_ID}')
print(f'Parallel envs   : {NUM_ENVS}')
print(f'PPO iterations  : {MAX_ITERATIONS}')
print(f'Policy freq     : {1 / (SIM_TIMESTEP * DECIMATION):.0f} Hz')
print(f'Episode length  : {EPISODE_LENGTH_S:.2f} s')
print(f'Outputs dir     : {OUT.resolve()}')
print(f'Media dir       : {MEDIA.resolve()}')
print(f'Log root        : {Path(LOG_ROOT).resolve()}')


## Training Setup

Build TrainConfig from the registered task and apply notebook overrides. Same pattern as `01_walk.ipynb` (uses `TrainConfig.from_task` + `run_train` — not subprocess).

In [ ]:
import datetime, os
from pathlib import Path
from mjlab.scripts.train import TrainConfig

cfg = TrainConfig.from_task(TASK_ID)
cfg.env.scene.num_envs    = NUM_ENVS
cfg.agent.max_iterations  = MAX_ITERATIONS
cfg.agent.logger          = 'tensorboard'
cfg.agent.experiment_name = EXPERIMENT_HINT
cfg.agent.resume          = False  # handled explicitly in runtrain cell

log_dir = (
    Path(LOG_ROOT) / EXPERIMENT_HINT
    / datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
)

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ.setdefault('MUJOCO_GL', 'egl')

_mode = 'RESUME' if RESUME else 'FROM SCRATCH'
print(f'Task            : {TASK_ID}')
print(f'Envs / iters    : {cfg.env.scene.num_envs} / {cfg.agent.max_iterations}')
print(f'Logger          : {cfg.agent.logger}')
print(f'CUDA device     : {os.environ["CUDA_VISIBLE_DEVICES"]}')
print(f'Mode            : {_mode}')
print(f'Log dir         : {log_dir.resolve()}')


## Run Training

Blocks the kernel until PPO training finishes. Use "Interrupt Kernel" to stop.

In [ ]:
import sys, os
from pathlib import Path
for _p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_p / 'LICENSE').exists():
        os.chdir(_p); break
sys.path.insert(0, str(Path.cwd()))
from src.utils import find_latest_checkpoint
from src.training.train_loop import run as run_train_loop
from mjlab.utils.os import get_checkpoint_path

_existing_ckpt = find_latest_checkpoint(LOG_ROOT, experiment_hint=EXPERIMENT_HINT)
if REUSE_TRAINING and _existing_ckpt is not None:
    print(f'[REUSE] Checkpoint found — skipping training.\n        {_existing_ckpt}')
else:
    resume_ckpt = None
    if RESUME:
        resume_ckpt = get_checkpoint_path(
            log_dir.parent, r'\d{4}-\d{2}-\d{2}_.*', 'model_.*.pt',
        )

    run_train_loop(
        task_id=TASK_ID,
        cfg=cfg,
        log_dir=log_dir,
        resume_ckpt=resume_ckpt,
    )


## Training Curves & Robustness

Post-training analysis in two blocks: (1) **training curves** parsed from the TensorBoard event file — reward, PPO diagnostics, per-term reward decomposition, terminations and command tracking; (2) **robustness** — headless rollouts of the trained policy under lateral pushes and a commanded-velocity sweep, following the RSL "Learning to Walk in Minutes" evaluation protocol (Rudin et al., CoRL 2022).

Set `RUN_ROBUSTNESS = False` in the setup cell below to skip figures 5–6 (the rollout-based ones).

In [ ]:
# Plot style (Wong palette + IEEEtran-like serif) and TB helpers.
import sys
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

_root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'LICENSE').exists()); import os as _os; _os.chdir(_root); sys.path.insert(0, str(_root))
from src.utils import find_latest_tb_log, find_latest_checkpoint

RUN_ROBUSTNESS = True  # False skips Figs 5-6 (rollout-based)

COLORS = {
    'primary':   '#0072B2',   # Wong blue
    'accent':    '#D55E00',   # Wong vermillion
    'ok':        '#009E73',   # Wong bluish-green
    'warn':      '#E69F00',   # Wong orange
    'neutral':   '#56B4E9',
    'text':      '#1A1A1A',
    'grid':      '#B0B0B0',
}

matplotlib.rcParams.update({
    'text.usetex': False,
    'font.family': 'serif',
    'font.serif': ['Times', 'Nimbus Roman', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'font.size': 9, 'axes.labelsize': 9, 'legend.fontsize': 7.5,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.linewidth': 0.6, 'axes.edgecolor': COLORS['text'],
    'xtick.direction': 'in', 'ytick.direction': 'in',
    'xtick.top': True, 'ytick.right': True,
    'xtick.minor.visible': True, 'ytick.minor.visible': True,
    'grid.color': COLORS['grid'], 'grid.linewidth': 0.4, 'grid.alpha': 0.35,
    'figure.dpi': 120,
})

def load_scalar(log_dir, tag):
    """Load a TensorBoard scalar series by tag.
    Args: log_dir — path to the run directory; tag — scalar tag name.
    Returns: tuple (steps, values) as numpy arrays; empty arrays if tag missing.
    """
    ea = EventAccumulator(str(log_dir)); ea.Reload()
    if tag not in ea.Tags().get('scalars', []):
        return np.array([]), np.array([])
    evs = ea.Scalars(tag)
    return (np.array([e.step for e in evs], dtype=float),
            np.array([e.value for e in evs], dtype=float))

def list_scalars(log_dir, prefix=''):
    """List TensorBoard scalar tags filtered by a name prefix.
    Args: log_dir — path to the run directory; prefix — string prefix filter.
    Returns: list of matching tag names.
    """
    ea = EventAccumulator(str(log_dir)); ea.Reload()
    return [t for t in ea.Tags().get('scalars', []) if t.startswith(prefix)]

def smooth(x, w=15):
    """Moving-average smoothing with edge padding.
    Args: x — 1-D input array; w — window width.
    Returns: smoothed array of the same length as x.
    """
    if len(x) < w or w < 2:
        return np.asarray(x, dtype=float)
    pad = w // 2
    xp = np.pad(x, pad, mode='edge')
    return np.convolve(xp, np.ones(w) / w, mode='valid')[:len(x)]

def apply_grid(ax):
    """Apply a light two-tier grid style to a matplotlib axis.
    Args: ax — matplotlib Axes.
    Returns: none.
    """
    ax.grid(True, which='major', linewidth=0.5, alpha=0.45)
    ax.grid(True, which='minor', linewidth=0.3, alpha=0.60)
    for s in ax.spines.values():
        s.set_linewidth(0.6)
def apply_grid(ax):
    """Apply a denser two-tier grid with auto minor locators (overrides the first).
    Args: ax — matplotlib Axes.
    Returns: none.
    """
    ax.minorticks_on()
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.grid(True, which='major', linewidth=0.5, alpha=0.80)
    ax.grid(True, which='minor', linewidth=0.3, alpha=0.70)
    for s in ax.spines.values():
        s.set_linewidth(0.6)
TB_DIR = find_latest_tb_log(LOG_ROOT, experiment_hint=EXPERIMENT_HINT)
assert TB_DIR is not None, 'No TensorBoard log found — train first.'
print(f'TB dir : {TB_DIR}')


## Figure 1: training progress and PPO diagnostics

2x3 grid of TensorBoard scalars — mean episode reward, episode length, reward gain
`dR/di`, entropy, value loss (log scale), and policy std. Reads from the latest run
under `outputs/logs/g1_reach/`. Saved to `outputs/media/walk_fig1_training.png`.

In [ ]:
# Fig 1: return + PPO diagnostics — 2x3 grid (reward, ep length, dR / entropy, value loss, policy std).
fig, axes = plt.subplots(2, 3, figsize=(9.6, 5.2))

# Row 1 — training progress.
x_r, y_r = load_scalar(TB_DIR, 'Train/mean_reward')
if len(y_r):
    axes[0, 0].plot(x_r, smooth(y_r), color=COLORS['primary'], lw=1.2)
axes[0, 0].set_xlabel('PPO iteration')
axes[0, 0].set_ylabel('Mean episode reward')
apply_grid(axes[0, 0])

x, y = load_scalar(TB_DIR, 'Train/mean_episode_length')
if len(y):
    axes[0, 1].plot(x, smooth(y), color=COLORS['ok'], lw=1.2)
axes[0, 1].axhline(1000, ls=':', lw=0.6, color=COLORS['text'], alpha=0.6)
axes[0, 1].text(x[-1] if len(x) else 0, 1010, 'max horizon',
                ha='right', va='bottom', fontsize=7, color=COLORS['text'], alpha=0.7)
axes[0, 1].set_xlabel('PPO iteration')
axes[0, 1].set_ylabel('Mean episode length (steps)')
apply_grid(axes[0, 1])

# Reward gain per iteration (dR/di) — how fast the policy is still improving.
if len(y_r) > 1:
    r_smooth = smooth(y_r)
    dr = np.gradient(r_smooth, x_r)
    axes[0, 2].plot(x_r, smooth(dr, w=25), color=COLORS['warn'], lw=1.2)
    axes[0, 2].axhline(0, ls=':', lw=0.6, color=COLORS['text'], alpha=0.6)
axes[0, 2].set_xlabel('PPO iteration')
axes[0, 2].set_ylabel(r'Reward gain $dR/di$')
apply_grid(axes[0, 2])

# Row 2 — PPO diagnostics.
for ax, tag, ylabel, color, log in [
    (axes[1, 0], 'Loss/entropy',    r'Entropy $H(\pi)$',        COLORS['primary'], False),
    (axes[1, 1], 'Loss/value',      r'Value loss $L_V$',        COLORS['accent'],  True),
    (axes[1, 2], 'Policy/mean_std', r'Policy std $\sigma_\pi$', COLORS['ok'],      False),
]:
    x, y = load_scalar(TB_DIR, tag)
    if len(y):
        ax.plot(x, smooth(y), color=color, lw=1.2)
    ax.set_xlabel('PPO iteration')
    ax.set_ylabel(ylabel)
    if log:
        ax.set_yscale('log')
    apply_grid(ax)

fig.tight_layout()
plt.savefig(MEDIA / 'walk_fig1_training.png', dpi=150, bbox_inches='tight')
plt.show()

if len(y_r):
    print(f'Peak reward: {np.max(smooth(y_r)):.2f}')


## Run Viewer

Blocks the kernel while the viewer is open. Use "Interrupt Kernel" to stop.

In [ ]:
import sys, os
from pathlib import Path
for _p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_p / 'LICENSE').exists():
        os.chdir(_p); break
sys.path.insert(0, str(Path.cwd()))
from src.utils import find_latest_checkpoint
from mjlab.scripts.play import PlayConfig, run_play

PLAY_AGENT          = 'trained'   # @param ["trained","random","zero"]
PLAY_NUM_ENVS       = 1           # @param {type:"integer"}
PLAY_VIEWER         = 'viser'     # @param ["viser","native"]
CHECKPOINT_OVERRIDE = ''          # @param {type:"string"}  leave blank for latest

checkpoint = (
    Path(CHECKPOINT_OVERRIDE)
    if CHECKPOINT_OVERRIDE.strip()
    else find_latest_checkpoint(LOG_ROOT, experiment_hint=EXPERIMENT_HINT)
)

if PLAY_AGENT == 'trained' and checkpoint is None:
    raise RuntimeError('No checkpoint found. Train first, or set CHECKPOINT_OVERRIDE.')

play_cfg = PlayConfig(
    agent=PLAY_AGENT,
    checkpoint_file=str(checkpoint) if (PLAY_AGENT == 'trained' and checkpoint) else None,
    num_envs=PLAY_NUM_ENVS,
    viewer=PLAY_VIEWER,
)

print(f'Agent      : {PLAY_AGENT}')
print(f'Viewer     : {PLAY_VIEWER}')
print(f'Num envs   : {PLAY_NUM_ENVS}')
if checkpoint:
    print(f'Checkpoint : {checkpoint}')

if RUN_VIEWER:
    print('Viewer starting — open http://localhost:8080 in your browser (viser).')
    print('Press Interrupt Kernel in the toolbar to close it.')
    run_play(TASK_ID, play_cfg)
else:
    print('Set RUN_VIEWER=True to open the trained policy.')